<a href="https://colab.research.google.com/github/TediBalint/AI-Jegyzetek/blob/master/Computer%20Vision/K%C3%A9paugment%C3%A1ci%C3%B3s%20Technik%C3%A1k.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Képaugmentációs Technikák

Az **adataugmentáció** a deep learning egyik legfontosabb regularizációs technikája. A tanító adatok mesterséges bővítésével csökkentjük a túltanulást és javítjuk a modell általánosító képességét.

## Tartalomjegyzék

1. Augmentáció alapjai
2. Geometriai transzformációk
3. Szín és intenzitás manipuláció
4. Speciális technikák (Cutout, Mixup, CutMix)
5. Automatikus augmentáció
6. Best Practices

## 1. Augmentáció alapjai

### Miért fontos?

- **Több adat**: Virtuálisan növeli a tanító adathalmaz méretét
- **Regularizáció**: Csökkenti a túltanulást
- **Invariancia tanulás**: A modell megtanulja figyelmen kívül hagyni az irreleváns variációkat
- **Robusztusság**: Valós körülmények között is jól működik

### Augmentáció típusok

| Típus | Példák | Alkalmazás |
|-------|--------|------------|
| Geometriai | Forgatás, tükrözés, nyírás | Pozíció-invariancia |
| Fotometriai | Fényerő, kontraszt, szín | Megvilágítás-invariancia |
| Noise | Gaussian noise, blur | Robusztusság |
| Speciális | Cutout, Mixup, CutMix | Regularizáció |

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFilter
from typing import Tuple, List, Callable
import random

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'DejaVu Sans'
np.random.seed(42)
torch.manual_seed(42)

# Szintetikus teszt kép létrehozása
def create_sample_image(size: int = 224) -> Image.Image:
    """Egyszerű teszt kép geometriai alakzatokkal."""
    img = Image.new('RGB', (size, size), color=(240, 240, 250))
    draw = ImageDraw.Draw(img)

    # Piros kör
    draw.ellipse([50, 50, 120, 120], fill=(220, 60, 60))
    # Zöld négyzet
    draw.rectangle([130, 80, 190, 140], fill=(60, 180, 60))
    # Kék háromszög
    draw.polygon([(100, 150), (60, 200), (140, 200)], fill=(60, 100, 220))
    # Sárga csillag
    cx, cy, r = 170, 180, 25
    points = []
    for i in range(10):
        angle = i * 36 - 90
        rad = r if i % 2 == 0 else r * 0.4
        x = cx + rad * np.cos(np.radians(angle))
        y = cy + rad * np.sin(np.radians(angle))
        points.append((x, y))
    draw.polygon(points, fill=(240, 200, 60))

    return img

# Teszt kép
sample_img = create_sample_image()
plt.figure(figsize=(4, 4))
plt.imshow(sample_img)
plt.title('Eredeti kép')
plt.axis('off')
plt.show()

## 2. Geometriai transzformációk

### Alapvető geometriai műveletek:
- **Flip**: Vízszintes/függőleges tükrözés
- **Rotation**: Forgatás tetszőleges szöggel
- **Scale/Crop**: Nagyítás, kivágás
- **Shear**: Nyírás
- **Affine**: Általános affin transzformáció

In [ ]:
def show_augmentations(img: Image.Image, transforms_dict: dict,
                       title: str = "Augmentációk"):
    """Transzformációk vizualizálása."""
    n = len(transforms_dict)
    cols = min(4, n)
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
    axes = np.array(axes).flatten() if n > 1 else [axes]

    for ax, (name, transform) in zip(axes, transforms_dict.items()):
        if transform is None:
            aug_img = img
        else:
            aug_img = transform(img)
        ax.imshow(aug_img)
        ax.set_title(name, fontsize=11)
        ax.axis('off')

    # Üres panelek elrejtése
    for ax in axes[n:]:
        ax.axis('off')

    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Geometriai transzformációk
geometric_transforms = {
    'Eredeti': None,
    'H-Flip': T.RandomHorizontalFlip(p=1.0),
    'V-Flip': T.RandomVerticalFlip(p=1.0),
    'Rotate 30°': T.RandomRotation(degrees=(30, 30)),
    'Rotate 90°': T.RandomRotation(degrees=(90, 90)),
    'Random Crop': T.RandomResizedCrop(224, scale=(0.6, 0.8)),
    'Center Crop': T.CenterCrop(150),
    'Affine': T.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1), shear=10),
}

show_augmentations(sample_img, geometric_transforms, "Geometriai transzformációk")

In [ ]:
def visualize_random_augmentation(img: Image.Image, transform: Callable,
                                   name: str, n_samples: int = 8):
    """Ugyanazon transzformáció többszöri alkalmazása."""
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))

    for i, ax in enumerate(axes.flat):
        aug_img = transform(img)
        ax.imshow(aug_img)
        ax.set_title(f'Minta {i+1}')
        ax.axis('off')

    plt.suptitle(f'{name} - Random variációk', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# RandomResizedCrop variációk
random_crop = T.RandomResizedCrop(224, scale=(0.5, 1.0), ratio=(0.8, 1.2))
visualize_random_augmentation(sample_img, random_crop, "RandomResizedCrop")

## 3. Szín és intenzitás manipuláció

### Fotometriai transzformációk:
- **Brightness**: Fényerő változtatása
- **Contrast**: Kontraszt állítása
- **Saturation**: Telítettség módosítása
- **Hue**: Színárnyalat eltolása
- **ColorJitter**: Kombinált szín-manipuláció

In [ ]:
# Fotometriai transzformációk
color_transforms = {
    'Eredeti': None,
    'Brightness+': T.ColorJitter(brightness=(1.5, 1.5)),
    'Brightness-': T.ColorJitter(brightness=(0.5, 0.5)),
    'Contrast+': T.ColorJitter(contrast=(2.0, 2.0)),
    'Saturation+': T.ColorJitter(saturation=(2.0, 2.0)),
    'Saturation-': T.ColorJitter(saturation=(0.2, 0.2)),
    'Hue shift': T.ColorJitter(hue=(0.3, 0.3)),
    'Grayscale': T.Grayscale(num_output_channels=3),
}

show_augmentations(sample_img, color_transforms, "Szín transzformációk")

In [ ]:
# ColorJitter kombinált hatása
color_jitter = T.ColorJitter(
    brightness=0.4,
    contrast=0.4,
    saturation=0.4,
    hue=0.1
)

visualize_random_augmentation(sample_img, color_jitter, "ColorJitter")

In [ ]:
# Noise és blur transzformációk
class GaussianNoise:
    """Gaussian zaj hozzáadása."""
    def __init__(self, mean: float = 0, std: float = 25):
        self.mean = mean
        self.std = std

    def __call__(self, img: Image.Image) -> Image.Image:
        img_array = np.array(img).astype(np.float32)
        noise = np.random.normal(self.mean, self.std, img_array.shape)
        noisy = np.clip(img_array + noise, 0, 255).astype(np.uint8)
        return Image.fromarray(noisy)

class RandomBlur:
    """Random Gaussian blur."""
    def __init__(self, radius_range: Tuple[float, float] = (0.5, 2.0)):
        self.radius_range = radius_range

    def __call__(self, img: Image.Image) -> Image.Image:
        radius = random.uniform(*self.radius_range)
        return img.filter(ImageFilter.GaussianBlur(radius=radius))

noise_transforms = {
    'Eredeti': None,
    'Gaussian Noise (σ=15)': GaussianNoise(std=15),
    'Gaussian Noise (σ=30)': GaussianNoise(std=30),
    'Gaussian Blur': RandomBlur((2.0, 2.0)),
}

show_augmentations(sample_img, noise_transforms, "Noise és Blur")

## 4. Speciális technikák

### Modern augmentációs módszerek:

| Technika | Leírás | Előny |
|----------|--------|-------|
| **Cutout** | Véletlenszerű négyzet kitakarása | Részleges okkúzióra robusztus |
| **Mixup** | Két kép és címke lineáris kombinációja | Simább döntési határok |
| **CutMix** | Kivágott régió másik képről | Cutout + Mixup előnyei |

In [ ]:
class Cutout:
    """Random négyzet alakú régió kitakarása."""
    def __init__(self, n_holes: int = 1, hole_size: int = 50, fill_value: int = 0):
        self.n_holes = n_holes
        self.hole_size = hole_size
        self.fill_value = fill_value

    def __call__(self, img: Image.Image) -> Image.Image:
        img_array = np.array(img).copy()
        h, w = img_array.shape[:2]

        for _ in range(self.n_holes):
            y = np.random.randint(0, h)
            x = np.random.randint(0, w)

            y1 = max(0, y - self.hole_size // 2)
            y2 = min(h, y + self.hole_size // 2)
            x1 = max(0, x - self.hole_size // 2)
            x2 = min(w, x + self.hole_size // 2)

            img_array[y1:y2, x1:x2] = self.fill_value

        return Image.fromarray(img_array)

# Cutout variációk
cutout_transforms = {
    'Eredeti': None,
    '1 hole (50px)': Cutout(n_holes=1, hole_size=50),
    '1 hole (80px)': Cutout(n_holes=1, hole_size=80),
    '3 holes (40px)': Cutout(n_holes=3, hole_size=40),
}

show_augmentations(sample_img, cutout_transforms, "Cutout")

In [ ]:
def mixup(img1: np.ndarray, img2: np.ndarray, label1: int, label2: int,
          alpha: float = 0.4) -> Tuple[np.ndarray, np.ndarray]:
    """Mixup: két kép és címke lineáris kombinációja."""
    lam = np.random.beta(alpha, alpha)
    mixed_img = (lam * img1 + (1 - lam) * img2).astype(np.uint8)
    mixed_label = np.array([lam, 1-lam])  # Soft label
    return mixed_img, mixed_label, lam

def cutmix(img1: np.ndarray, img2: np.ndarray, label1: int, label2: int,
           alpha: float = 1.0) -> Tuple[np.ndarray, np.ndarray]:
    """CutMix: egy régió cseréje másik képről."""
    lam = np.random.beta(alpha, alpha)
    h, w = img1.shape[:2]

    # Vágási régió mérete
    cut_ratio = np.sqrt(1 - lam)
    cut_h = int(h * cut_ratio)
    cut_w = int(w * cut_ratio)

    # Véletlenszerű pozíció
    cx = np.random.randint(0, w)
    cy = np.random.randint(0, h)

    x1 = max(0, cx - cut_w // 2)
    x2 = min(w, cx + cut_w // 2)
    y1 = max(0, cy - cut_h // 2)
    y2 = min(h, cy + cut_h // 2)

    mixed_img = img1.copy()
    mixed_img[y1:y2, x1:x2] = img2[y1:y2, x1:x2]

    # Tényleges arány
    lam_actual = 1 - (x2 - x1) * (y2 - y1) / (h * w)
    mixed_label = np.array([lam_actual, 1-lam_actual])

    return mixed_img, mixed_label, (x1, y1, x2, y2)

# Második kép létrehozása
def create_second_image(size: int = 224) -> Image.Image:
    img = Image.new('RGB', (size, size), color=(250, 245, 230))
    draw = ImageDraw.Draw(img)
    draw.ellipse([80, 80, 150, 150], fill=(180, 80, 200))  # Lila kör
    draw.rectangle([40, 40, 80, 80], fill=(80, 200, 200))  # Türkiz négyzet
    return img

img1 = np.array(sample_img)
img2 = np.array(create_second_image())

# Mixup és CutMix demonstráció
fig, axes = plt.subplots(2, 4, figsize=(14, 7))

# Eredeti képek
axes[0, 0].imshow(img1)
axes[0, 0].set_title('Kép 1 (Label: A)')
axes[0, 1].imshow(img2)
axes[0, 1].set_title('Kép 2 (Label: B)')

# Mixup
mixed, label, lam = mixup(img1, img2, 0, 1)
axes[0, 2].imshow(mixed)
axes[0, 2].set_title(f'Mixup (λ={lam:.2f})\nLabel: {lam:.2f}A + {1-lam:.2f}B')

mixed2, label2, lam2 = mixup(img1, img2, 0, 1)
axes[0, 3].imshow(mixed2)
axes[0, 3].set_title(f'Mixup (λ={lam2:.2f})\nLabel: {lam2:.2f}A + {1-lam2:.2f}B')

# CutMix
axes[1, 0].imshow(img1)
axes[1, 0].set_title('Kép 1')
axes[1, 1].imshow(img2)
axes[1, 1].set_title('Kép 2')

cutmixed, label_cm, bbox = cutmix(img1, img2, 0, 1)
axes[1, 2].imshow(cutmixed)
axes[1, 2].set_title(f'CutMix\nLabel: {label_cm[0]:.2f}A + {label_cm[1]:.2f}B')

cutmixed2, label_cm2, bbox2 = cutmix(img1, img2, 0, 1)
axes[1, 3].imshow(cutmixed2)
axes[1, 3].set_title(f'CutMix\nLabel: {label_cm2[0]:.2f}A + {label_cm2[1]:.2f}B')

for ax in axes.flat:
    ax.axis('off')

plt.suptitle('Mixup vs CutMix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Automatikus augmentáció

### AutoAugment és RandAugment

Az augmentáció kiválasztása és erőssége is tanulható vagy automatizálható!

- **AutoAugment**: Reinforcement learning-gel optimalizált policy
- **RandAugment**: Egyszerűbb, random kiválasztás fix erősséggel

In [ ]:
class RandAugment:
    """Egyszerűsített RandAugment implementáció."""

    def __init__(self, n_ops: int = 2, magnitude: int = 9):
        self.n_ops = n_ops
        self.magnitude = magnitude  # 0-10 skálán

        # Elérhető műveletek
        self.operations = [
            self._rotate,
            self._shear_x,
            self._shear_y,
            self._translate_x,
            self._translate_y,
            self._brightness,
            self._contrast,
            self._saturation,
            self._sharpness,
            self._posterize,
        ]

    def _rotate(self, img, m):
        angle = (m / 10) * 30  # max 30°
        return TF.rotate(img, angle * random.choice([-1, 1]))

    def _shear_x(self, img, m):
        shear = (m / 10) * 0.3
        return TF.affine(img, 0, (0, 0), 1.0, (shear * random.choice([-1, 1]), 0))

    def _shear_y(self, img, m):
        shear = (m / 10) * 0.3
        return TF.affine(img, 0, (0, 0), 1.0, (0, shear * random.choice([-1, 1])))

    def _translate_x(self, img, m):
        translate = int((m / 10) * img.size[0] * 0.3)
        return TF.affine(img, 0, (translate * random.choice([-1, 1]), 0), 1.0, (0, 0))

    def _translate_y(self, img, m):
        translate = int((m / 10) * img.size[1] * 0.3)
        return TF.affine(img, 0, (0, translate * random.choice([-1, 1])), 1.0, (0, 0))

    def _brightness(self, img, m):
        factor = 1 + (m / 10) * 0.5 * random.choice([-1, 1])
        return TF.adjust_brightness(img, max(0.1, factor))

    def _contrast(self, img, m):
        factor = 1 + (m / 10) * 0.5 * random.choice([-1, 1])
        return TF.adjust_contrast(img, max(0.1, factor))

    def _saturation(self, img, m):
        factor = 1 + (m / 10) * 0.5 * random.choice([-1, 1])
        return TF.adjust_saturation(img, max(0.1, factor))

    def _sharpness(self, img, m):
        factor = 1 + (m / 10) * 2 * random.choice([-1, 1])
        return TF.adjust_sharpness(img, max(0, factor))

    def _posterize(self, img, m):
        bits = 8 - int((m / 10) * 4)
        return TF.posterize(img, max(1, bits))

    def __call__(self, img: Image.Image) -> Image.Image:
        ops = random.sample(self.operations, self.n_ops)
        for op in ops:
            img = op(img, self.magnitude)
        return img

# RandAugment különböző erősségekkel
fig, axes = plt.subplots(2, 4, figsize=(14, 7))

magnitudes = [0, 3, 6, 9]

for i, m in enumerate(magnitudes):
    ra = RandAugment(n_ops=2, magnitude=m)
    for j in range(2):
        aug_img = ra(sample_img)
        axes[j, i].imshow(aug_img)
        axes[j, i].set_title(f'Magnitude={m}')
        axes[j, i].axis('off')

plt.suptitle('RandAugment - Különböző erősségek (n=2 művelet)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# PyTorch beépített augmentációk - tipikus pipeline-ok
def create_augmentation_pipelines():
    """Különböző komplexitású augmentációs pipeline-ok."""

    # Alap (gyenge)
    basic = T.Compose([
        T.RandomHorizontalFlip(p=0.5),
        T.ToTensor(),
    ])

    # Közepes
    medium = T.Compose([
        T.RandomHorizontalFlip(p=0.5),
        T.RandomRotation(15),
        T.ColorJitter(brightness=0.2, contrast=0.2),
        T.ToTensor(),
    ])

    # Erős
    strong = T.Compose([
        T.RandomResizedCrop(224, scale=(0.7, 1.0)),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomRotation(20),
        T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
        T.RandomGrayscale(p=0.1),
        T.ToTensor(),
    ])

    # ImageNet standard
    imagenet = T.Compose([
        T.RandomResizedCrop(224),
        T.RandomHorizontalFlip(),
        T.AutoAugment(T.AutoAugmentPolicy.IMAGENET),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    return {'Basic': basic, 'Medium': medium, 'Strong': strong}

pipelines = create_augmentation_pipelines()

fig, axes = plt.subplots(3, 4, figsize=(14, 10))

for i, (name, pipeline) in enumerate(pipelines.items()):
    for j in range(4):
        # Pipeline alkalmazása (ToTensor-ig, normalizálás nélkül vizualizációhoz)
        # Egyszerűsített verzió
        if name == 'Basic':
            t = T.Compose([T.RandomHorizontalFlip(p=0.5)])
        elif name == 'Medium':
            t = T.Compose([T.RandomHorizontalFlip(p=0.5), T.RandomRotation(15),
                          T.ColorJitter(brightness=0.2, contrast=0.2)])
        else:
            t = T.Compose([T.RandomResizedCrop(224, scale=(0.7, 1.0)),
                          T.RandomHorizontalFlip(p=0.5), T.RandomRotation(20),
                          T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1)])

        aug_img = t(sample_img)
        axes[i, j].imshow(aug_img)
        axes[i, j].axis('off')
        if j == 0:
            axes[i, j].set_ylabel(name, fontsize=12, fontweight='bold')

plt.suptitle('Augmentációs pipeline-ok összehasonlítása', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Best Practices

### Augmentáció ajánlások feladattípusonként:

| Feladat | Ajánlott | Kerülendő |
|---------|----------|------------|
| Általános osztályozás | Flip, Crop, Color | Túl erős torzítás |
| Orvosi képek | Rotation, Scale | Szín manipuláció |
| Arcfelismerés | Affine, Lighting | Erős forgatás |
| OCR/Szöveg | Shear, Noise | V-Flip |
| Műholdképek | Rotation, Scale | H-Flip (orientáció fontos) |

In [ ]:
def create_augmentation_summary():
    """Augmentáció összefoglaló vizualizáció."""

    fig, ax = plt.subplots(figsize=(14, 8))

    categories = {
        'Geometriai': ['HFlip', 'VFlip', 'Rotate', 'Crop', 'Scale', 'Shear', 'Affine'],
        'Fotometriai': ['Brightness', 'Contrast', 'Saturation', 'Hue', 'Grayscale'],
        'Noise/Blur': ['Gaussian', 'Blur', 'Sharpen'],
        'Speciális': ['Cutout', 'Mixup', 'CutMix', 'RandAugment'],
    }

    colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']

    y_pos = 0.9
    for i, (cat, items) in enumerate(categories.items()):
        # Kategória címke
        ax.text(0.02, y_pos, cat, fontsize=12, fontweight='bold', color=colors[i])

        # Elemek
        x_pos = 0.18
        for item in items:
            rect = plt.Rectangle((x_pos, y_pos - 0.03), 0.1, 0.06,
                                 facecolor=colors[i], alpha=0.3, edgecolor=colors[i])
            ax.add_patch(rect)
            ax.text(x_pos + 0.05, y_pos, item, ha='center', va='center', fontsize=9)
            x_pos += 0.11

        y_pos -= 0.15

    # Tippek
    tips = [
        "✓ Mindig validációs adaton augmentáció NÉLKÜL értékelj",
        "✓ Kezdj gyenge augmentációval, fokozatosan erősítsd",
        "✓ Domain-specifikus augmentációt válassz",
        "✓ Mixup/CutMix különösen hatékony nagy modelleknél",
        "✗ Ne alkalmazz olyan transzformációt, ami megváltoztatja a címkét",
    ]

    y_pos = 0.3
    ax.text(0.02, y_pos + 0.05, 'Best Practices:', fontsize=12, fontweight='bold')
    for tip in tips:
        color = 'green' if tip.startswith('✓') else 'red'
        ax.text(0.02, y_pos, tip, fontsize=10, color=color)
        y_pos -= 0.06

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.set_title('Image Augmentation Összefoglaló', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

create_augmentation_summary()

## Összefoglalás

### Augmentáció típusok:

| Kategória | Technikák | Használat |
|-----------|-----------|------------|
| Geometriai | Flip, Rotate, Crop, Affine | Pozíció-invariancia |
| Fotometriai | ColorJitter, Grayscale | Megvilágítás-robusztusság |
| Noise | Gaussian, Blur | Zaj-tolerancia |
| Speciális | Cutout, Mixup, CutMix | Erős regularizáció |

### Tipikus pipeline:
```python
train_transform = T.Compose([
    T.RandomResizedCrop(224),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.4, 0.4, 0.4, 0.1),
    T.ToTensor(),
    T.Normalize(mean, std),
])
```

### Kulcs tanulságok:
- Az augmentáció **ingyenes** regularizáció
- **Domain-specifikus** megközelítés szükséges
- **Validáció** mindig augmentáció nélkül
- Modern módszerek (Mixup, CutMix) **jelentős javulást** hoznak